# OmniDiag — Heart Module: Training, Validation & Evidence**What this notebook produces**, all under `/kaggle/working/`:| Output | File ||---|---|| Production model (full 11 features) | `models/heart_full.pkl` || Transfer model (7 shared features) | `models/heart_reduced.pkl` || Final metrics table | `evidence/final_metrics.json` || Leave-one-site-out results | `evidence/loso.json` || External validation (Tehran) | `evidence/external_validation.json` || Threshold decision log | `evidence/threshold_log.json` || Charts | `figures/*.png` |**Validation design**1. `pooled_cv` — 5-fold CV with sites mixed. Optimistic; reported only as a reference point.2. `LOSO` — each of the four UCI hospitals held out entirely. This is the honest internal number.3. `external` — trained on all UCI sites, tested on Z-Alizadeh Sani (Tehran). Never seen in training.**Rules enforced throughout**- Every preprocessing step (imputation, encoding, scaling) lives inside the `Pipeline`, so it is fitted on training folds only.- Decision thresholds are selected on out-of-fold predictions of the **training** data. No test set ever informs a threshold.- Nothing is imputed with a constant to make a schema line up. Features without a genuine source are dropped.**Required Kaggle dataset inputs:** `uci_heart_by_site.csv`, `z_alizadeh_translated.csv`, `z_alizadeh_mapping.json`

In [ ]:
!pip -q install xgboost shap --upgrade 2>/dev/nullimport json, os, warnings, globwarnings.filterwarnings("ignore")import numpy as np, pandas as pd, xgboost as xgbimport matplotlib; matplotlib.use("Agg")import matplotlib.pyplot as pltimport joblibfrom sklearn.compose import ColumnTransformerfrom sklearn.experimental import enable_iterative_imputer  # noqafrom sklearn.impute import IterativeImputer, SimpleImputerfrom sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve,                             classification_report)from sklearn.model_selection import StratifiedKFold, cross_val_predictfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import OrdinalEncoder, StandardScalerfrom sklearn.calibration import calibration_curveSEED, N_BOOT = 42, 2000FN_COST, FP_COST = 2.0, 1.0rng = np.random.default_rng(SEED)for d in ["models", "evidence", "figures"]:    os.makedirs(d, exist_ok=True)def find(name):    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) or glob.glob(name)    if not hits:        raise FileNotFoundError(f"{name} not found. Attach the dataset.")    return hits[0]uci = pd.read_csv(find("uci_heart_by_site.csv"))zal = pd.read_csv(find("z_alizadeh_translated.csv"))MAPPING = json.load(open(find("z_alizadeh_mapping.json")))print("UCI  ", uci.shape, "| sites:", list(uci.site.unique()))print("Tehran", zal.shape)print("versions: sklearn", __import__("sklearn").__version__, "xgboost", xgb.__version__)

## 1. Feature sets`FULL` is what the production API collects. `SHARED` is the subset that also exists,as a genuine measurement, in the Tehran cohort — the only set on which an externalcomparison is meaningful.

In [ ]:
NUM_FULL = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak", "FastingBS"]CAT_FULL = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]NUM_RED  = ["Age", "RestingBP", "Cholesterol", "FastingBS"]CAT_RED  = ["Sex", "ChestPainType", "RestingECG"]FULL, RED = NUM_FULL + CAT_FULL, NUM_RED + CAT_REDTARGET = "HeartDisease"PARAMS = dict(n_estimators=898, max_depth=5, learning_rate=0.013594126498405943,              subsample=0.963733407970185, colsample_bytree=0.5454718650965412,              random_state=SEED, eval_metric="logloss")def make_pipe(num, cat):    return Pipeline([        ("prep", ColumnTransformer([            ("num", Pipeline([("imp", IterativeImputer(random_state=SEED, max_iter=10)),                              ("sc", StandardScaler())]), num),            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),                              ("enc", OrdinalEncoder(handle_unknown="use_encoded_value",                                                     unknown_value=-1))]), cat),        ])),        ("clf", xgb.XGBClassifier(**PARAMS)),    ])cv = StratifiedKFold(5, shuffle=True, random_state=SEED)def metrics_at(y, proba, thr):    pred = (proba >= thr).astype(int)    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()    return dict(accuracy=(tp+tn)/len(y), roc_auc=roc_auc_score(y, proba),                sensitivity=tp/(tp+fn) if tp+fn else np.nan,                specificity=tn/(tn+fp) if tn+fp else np.nan,                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))def pick_threshold(y, proba, log=None):    grid, best, bc = np.linspace(0.01, 0.99, 200), 0.5, np.inf    scan = []    for t in grid:        tn, fp, fn, tp = confusion_matrix(y, (proba >= t).astype(int), labels=[0,1]).ravel()        c = FN_COST*fn + FP_COST*fp        scan.append(dict(threshold=round(float(t),4), cost=float(c),                         tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp)))        if c < bc: bc, best = c, float(t)    if log is not None:        log.update(grid=[0.01, 0.99, len(grid)],                   cost_function=f"{FN_COST}*FN + {FP_COST}*FP",                   selected=round(best,4), minimum_cost=bc, scan=scan)    return bestdef bootstrap_ci(y, proba, thr, n=N_BOOT):    y, proba = np.asarray(y), np.asarray(proba)    acc = {k: [] for k in ["accuracy","roc_auc","sensitivity","specificity"]}    for _ in range(n):        i = rng.integers(0, len(y), len(y))        if len(np.unique(y[i])) < 2: continue        m = metrics_at(y[i], proba[i], thr)        for k in acc: acc[k].append(m[k])    return {k: [round(float(np.percentile(v,2.5))*100,2),                round(float(np.percentile(v,97.5))*100,2)] for k,v in acc.items()}

## 2. Data auditPrinted before any modelling, because a reviewer will ask about it first:sample size per site, prevalence, missingness, and the clinical direction of thelabel at every site.

In [ ]:
rows = []for s, g in uci.groupby("site", sort=False):    rows.append(dict(site=s, n=len(g), disease=int(g[TARGET].sum()),                     prevalence=f"{g[TARGET].mean()*100:.1f}%",                     male=f"{(g.Sex=='M').mean()*100:.0f}%", age=round(g.Age.mean(),1)))rows.append(dict(site="z_alizadeh_tehran", n=len(zal), disease=int(zal[TARGET].sum()),                 prevalence=f"{zal[TARGET].mean()*100:.1f}%",                 male=f"{(zal.Sex=='M').mean()*100:.0f}%", age=round(zal.Age.mean(),1)))cohorts = pd.DataFrame(rows)display(cohorts)print("\nMissingness by site (%)")display(uci.groupby("site", sort=False)[FULL].apply(lambda g: (g.isna().mean()*100).round(1)).T)print("\nLabel direction (no-disease -> disease); expect age up, MaxHR down, Oldpeak up")for s, g in uci.groupby("site", sort=False):    a, b = g[g[TARGET]==1], g[g[TARGET]==0]    print(f"  {s:<16} age {b.Age.mean():5.1f}->{a.Age.mean():5.1f}   "          f"MaxHR {b.MaxHR.mean():6.1f}->{a.MaxHR.mean():6.1f}   "          f"Oldpeak {b.Oldpeak.mean():4.2f}->{a.Oldpeak.mean():4.2f}")a, b = zal[zal[TARGET]==1], zal[zal[TARGET]==0]print(f"  {'tehran':<16} age {b.Age.mean():5.1f}->{a.Age.mean():5.1f}   "      f"(MaxHR/Oldpeak not measured in this cohort)")cohorts.to_json("evidence/cohorts.json", orient="records", indent=2)

## 3. Model A — full feature setPooled CV first (the optimistic reference), then leave-one-site-out.The difference between them is the generalisation gap.

In [ ]:
X, y, site = uci[FULL], uci[TARGET], uci["site"]thr_log = {}oof = cross_val_predict(make_pipe(NUM_FULL, CAT_FULL), X, y, cv=cv,                        method="predict_proba")[:, 1]POOLED_THR = pick_threshold(y, oof, thr_log)pooled = metrics_at(y, oof, POOLED_THR)pooled_ci = bootstrap_ci(y, oof, POOLED_THR)print(f"Pooled 5-fold CV, n={len(y)}")print(f"  threshold (out-of-fold, cost {FN_COST}xFN+{FP_COST}xFP): {POOLED_THR:.4f}")for k in ["accuracy","roc_auc","sensitivity","specificity"]:    print(f"  {k:<12}{pooled[k]*100:6.2f}%   95% CI {pooled_ci[k]}")json.dump(thr_log, open("evidence/threshold_log.json","w"), indent=2)

In [ ]:
loso = {}for s in uci.site.unique():    tr, te = site != s, site == s    log_s = {}    oof_tr = cross_val_predict(make_pipe(NUM_FULL, CAT_FULL), X[tr], y[tr],                               cv=cv, method="predict_proba")[:, 1]    t_s = pick_threshold(y[tr], oof_tr, log_s)    pipe = make_pipe(NUM_FULL, CAT_FULL).fit(X[tr], y[tr])    p = pipe.predict_proba(X[te])[:, 1]    m = metrics_at(y[te], p, t_s)    loso[s] = dict(n=int(te.sum()), prevalence=round(y[te].mean()*100,1),                   threshold=round(t_s,4),                   **{k: round(m[k]*100,2) for k in                      ["accuracy","roc_auc","sensitivity","specificity"]},                   tn=m["tn"], fp=m["fp"], fn=m["fn"], tp=m["tp"],                   ci=bootstrap_ci(y[te], p, t_s),                   controls=int(m["tn"]+m["fp"]), cases=int(m["tp"]+m["fn"]))loso_auc = np.mean([v["roc_auc"] for v in loso.values()])/100gap = pooled["roc_auc"] - loso_aucdisplay(pd.DataFrame(loso).T[["n","prevalence","roc_auc","threshold","sensitivity","specificity"]])print(f"\nLOSO mean ROC-AUC {loso_auc:.4f}   pooled {pooled['roc_auc']:.4f}"      f"   generalisation gap {gap:+.4f}")print("Thresholds across training sets:",      [round(v['threshold'],3) for v in loso.values()],      "<- a fixed threshold does not transfer")json.dump(dict(pooled_cv=dict(threshold=round(POOLED_THR,4),                              **{k: round(pooled[k]*100,2) for k in                                 ["accuracy","roc_auc","sensitivity","specificity"]},                              ci=pooled_ci),               leave_one_site_out=loso,               loso_mean_auc=round(float(loso_auc),4),               generalisation_gap=round(float(gap),4)),          open("evidence/loso.json","w"), indent=2)

## 4. Model B — shared features only, and external validation on TehranFour features have no honest source in the Tehran cohort (`MaxHR`, `Oldpeak`,`ST_Slope`, `ExerciseAngina`). They are dropped from **both** sides rather thanfilled with a constant, so the comparison is like for like.The first number below is the price of dropping them; the second is the price ofmoving to a different country.

In [ ]:
for k, v in MAPPING.items():    print(f"  {k:<16}{v['kind']:<13}<- {v['source']}")print()oof_r = cross_val_predict(make_pipe(NUM_RED, CAT_RED), uci[RED], y, cv=cv,                          method="predict_proba")[:, 1]thr_r_log = {}THR_RED = pick_threshold(y, oof_r, thr_r_log)red_int = metrics_at(y, oof_r, THR_RED)print(f"Reduced model, UCI internal pooled CV : ROC-AUC {red_int['roc_auc']:.4f}")print(f"Full model,    UCI internal pooled CV : ROC-AUC {pooled['roc_auc']:.4f}")print(f"  cost of dropping 4 features         : {red_int['roc_auc']-pooled['roc_auc']:+.4f}\n")ext_pipe = make_pipe(NUM_RED, CAT_RED).fit(uci[RED], y)p_ext = ext_pipe.predict_proba(zal[RED])[:, 1]ext = metrics_at(zal[TARGET], p_ext, THR_RED)ext_ci = bootstrap_ci(zal[TARGET], p_ext, THR_RED)print(f"EXTERNAL VALIDATION — Tehran, n={len(zal)}, prevalence "      f"{zal[TARGET].mean()*100:.1f}%, threshold {THR_RED:.4f} (from UCI only)")for k in ["accuracy","roc_auc","sensitivity","specificity"]:    print(f"  {k:<12}{ext[k]*100:6.2f}%   95% CI {ext_ci[k]}")print(f"\n  external minus internal ROC-AUC: {ext['roc_auc']-red_int['roc_auc']:+.4f}")shift = pd.concat([uci.ChestPainType.value_counts(normalize=True).rename("UCI"),                   zal.ChestPainType.value_counts(normalize=True).rename("Tehran")],                  axis=1).round(3)print("\nChestPainType distribution shift (the main driver of the drop):")display(shift)json.dump(dict(shared_features=RED, dropped=[k for k,v in MAPPING.items()                                             if v["kind"]=="UNAVAILABLE"],               mapping=MAPPING,               reduced_internal={k: round(red_int[k]*100,2) for k in                                 ["accuracy","roc_auc","sensitivity","specificity"]},               external={**{k: round(ext[k]*100,2) for k in                            ["accuracy","roc_auc","sensitivity","specificity"]},                         "ci": ext_ci, "n": int(len(zal)),                         "threshold": round(THR_RED,4),                         "tn":ext["tn"],"fp":ext["fp"],"fn":ext["fn"],"tp":ext["tp"]},               chest_pain_shift=shift.to_dict()),          open("evidence/external_validation.json","w"), indent=2)json.dump(thr_r_log, open("evidence/threshold_log_reduced.json","w"), indent=2)

## 5. Charts

In [ ]:
plt.rcParams.update({"figure.dpi":140, "font.size":9})# --- ROC curves, one per held-out site + externalfig, ax = plt.subplots(figsize=(5.4,4.6))for s in uci.site.unique():    tr, te = site != s, site == s    pp = make_pipe(NUM_FULL, CAT_FULL).fit(X[tr], y[tr]).predict_proba(X[te])[:,1]    fpr, tpr, _ = roc_curve(y[te], pp)    ax.plot(fpr, tpr, lw=1.6, label=f"{s} (AUC {roc_auc_score(y[te],pp):.3f})")fpr, tpr, _ = roc_curve(zal[TARGET], p_ext)ax.plot(fpr, tpr, lw=2.2, ls="--", color="k",        label=f"Tehran external (AUC {ext['roc_auc']:.3f})")ax.plot([0,1],[0,1], lw=.8, color="grey")ax.set_xlabel("1 - specificity"); ax.set_ylabel("sensitivity")ax.set_title("Leave-one-site-out ROC + external validation")ax.legend(fontsize=7.5, loc="lower right"); fig.tight_layout()fig.savefig("figures/roc_by_site.png"); plt.close(fig)# --- AUC barfig, ax = plt.subplots(figsize=(5.4,3.2))names = list(loso)+["tehran\n(external)"]vals  = [loso[s]["roc_auc"]/100 for s in loso]+[ext["roc_auc"]]ax.bar(range(len(vals)), vals, color=["#4C78A8"]*len(loso)+["#333"])ax.axhline(pooled["roc_auc"], ls="--", color="crimson", lw=1,           label=f"pooled CV {pooled['roc_auc']:.3f}")ax.set_xticks(range(len(vals))); ax.set_xticklabels(names, fontsize=7)ax.set_ylim(0.5,1.0); ax.set_ylabel("ROC-AUC")ax.set_title("Optimism of pooled CV vs held-out sites")ax.legend(fontsize=7.5); fig.tight_layout()fig.savefig("figures/auc_by_site.png"); plt.close(fig)# --- threshold cost sweepscan = pd.DataFrame(thr_log["scan"])fig, ax = plt.subplots(figsize=(5.4,3.2))ax.plot(scan.threshold, scan.cost, lw=1.6)ax.axvline(POOLED_THR, color="crimson", ls="--", lw=1,           label=f"selected {POOLED_THR:.4f}")ax.axvline(0.5, color="grey", ls=":", lw=1, label="default 0.50")ax.set_xlabel("decision threshold"); ax.set_ylabel(f"{FN_COST}xFN + {FP_COST}xFP")ax.set_title("Threshold selected on out-of-fold training predictions")ax.legend(fontsize=7.5); fig.tight_layout()fig.savefig("figures/threshold_sweep.png"); plt.close(fig)# --- confusion matricesfig, axes = plt.subplots(1, 2, figsize=(7.6,3.4))for axi, (title, m) in zip(axes, [("UCI pooled CV", pooled), ("Tehran external", ext)]):    cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])    axi.imshow(cm, cmap="Blues")    for i in range(2):        for j in range(2):            axi.text(j, i, cm[i,j], ha="center", va="center", fontsize=14,                     color="white" if cm[i,j] > cm.max()/2 else "black")    axi.set_xticks([0,1], ["pred -","pred +"]); axi.set_yticks([0,1], ["true -","true +"])    axi.set_title(title, fontsize=9)fig.tight_layout(); fig.savefig("figures/confusion_matrices.png"); plt.close(fig)# --- calibrationfig, ax = plt.subplots(figsize=(4.4,4.0))for lbl, yy, pp in [("UCI out-of-fold", y, oof), ("Tehran external", zal[TARGET], p_ext)]:    a_, b_ = calibration_curve(yy, pp, n_bins=8, strategy="quantile")    ax.plot(b_, a_, "o-", lw=1.4, ms=4, label=lbl)ax.plot([0,1],[0,1], "--", color="grey", lw=.8)ax.set_xlabel("predicted probability"); ax.set_ylabel("observed frequency")ax.set_title("Calibration"); ax.legend(fontsize=7.5); fig.tight_layout()fig.savefig("figures/calibration.png"); plt.close(fig)print("figures written:", sorted(os.listdir("figures")))

## 6. Production models + SHAPBoth models are fitted on **all** UCI data and saved. The Tehran cohort is notused for fitting anywhere in this notebook — it exists only to report theexternal number above.

In [ ]:
full_model = make_pipe(NUM_FULL, CAT_FULL).fit(X, y)red_model  = make_pipe(NUM_RED,  CAT_RED ).fit(uci[RED], y)joblib.dump(dict(pipeline=full_model, features=FULL, numeric=NUM_FULL,                 categorical=CAT_FULL, threshold=round(POOLED_THR,4),                 trained_on="UCI 4 sites", n=int(len(y)),                 sklearn=__import__("sklearn").__version__, xgboost=xgb.__version__),            "models/heart_full.pkl")joblib.dump(dict(pipeline=red_model, features=RED, numeric=NUM_RED,                 categorical=CAT_RED, threshold=round(THR_RED,4),                 trained_on="UCI 4 sites (shared features)", n=int(len(y))),            "models/heart_reduced.pkl")print("saved:", os.listdir("models"))try:    import shap    Xt = full_model.named_steps["prep"].transform(X)    names = NUM_FULL + CAT_FULL    ex = shap.TreeExplainer(full_model.named_steps["clf"])    sv = ex.shap_values(Xt)    plt.figure()    shap.summary_plot(sv, pd.DataFrame(Xt, columns=names), show=False, plot_size=(6,4))    plt.tight_layout(); plt.savefig("figures/shap_summary.png", dpi=140); plt.close()    imp = pd.Series(np.abs(sv).mean(0), index=names).sort_values(ascending=False)    display(imp.round(4).to_frame("mean |SHAP|"))    imp.round(6).to_json("evidence/shap_importance.json", indent=2)except Exception as e:    print("SHAP skipped:", type(e).__name__, e)

## 7. Final metrics tableOne table. Every number in any document must come from here, and every row namesthe file that proves it.

In [ ]:
final = {  "generated_by": "OmniDiag heart notebook",  "training_data": {"source":"UCI Heart Disease, 4 original site files",                    "n": int(len(uci)),                    "sites": {s:int((uci.site==s).sum()) for s in uci.site.unique()},                    "note":"ca and thal excluded: invasive, absent from production"},  "external_data": {"source":"Z-Alizadeh Sani (Tehran)", "n": int(len(zal)),                    "used_for":"validation only, never training"},  "headline": {    "internal_pooled_cv": {"roc_auc": round(pooled["roc_auc"]*100,2),                           "ci95": pooled_ci["roc_auc"],                           "caveat":"sites mixed across folds; optimistic"},    "leave_one_site_out": {"mean_roc_auc": round(float(loso_auc)*100,2),                           "range":[round(min(v['roc_auc'] for v in loso.values()),2),                                    round(max(v['roc_auc'] for v in loso.values()),2)],                           "interpretation":"transfer to an unseen hospital"},    "external_tehran":    {"roc_auc": round(ext["roc_auc"]*100,2),                           "ci95": ext_ci["roc_auc"],                           "features": len(RED),                           "interpretation":"transfer to a different country"},    "generalisation_gap": round(float(gap)*100,2),  },  "operating_point": {    "threshold": round(POOLED_THR,4),    "selected_on":"out-of-fold predictions of training data",    "cost_function": f"{FN_COST}*FN + {FP_COST}*FP",    **{k:{"value":round(pooled[k]*100,2),"ci95":pooled_ci[k]} for k in       ["accuracy","sensitivity","specificity"]},    "confusion":{k:pooled[k] for k in ["tn","fp","fn","tp"]},  },  "evidence_files": sorted(glob.glob("evidence/*")+glob.glob("figures/*")+glob.glob("models/*")),}json.dump(final, open("evidence/final_metrics.json","w"), indent=2)print(json.dumps(final["headline"], indent=2))print(json.dumps(final["operating_point"], indent=2))

In [ ]:
!cd /kaggle/working && zip -qr omnidiag_heart_outputs.zip models evidence figures && ls -la omnidiag_heart_outputs.zip